# Guardar y Cargar Modelos en PyTorch

Este notebook demuestra cómo guardar y cargar modelos de PyTorch, cubriendo los diferentes métodos disponibles.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

## 1. Definir un Modelo Simple

Primero, definimos una red neuronal simple para nuestros ejemplos:

In [2]:
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc1 = nn.Linear(10, 50)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(50, 1)
        
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Crear una instancia del modelo
model = SimpleModel()

# Crear un optimizador
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Datos de entrada de ejemplo
dummy_input = torch.randn(1, 10)
dummy_output = model(dummy_input)

print("Modelo creado:")
print(model)
print(f"\nPredicción de ejemplo: {dummy_output.item():.4f}")

Modelo creado:
SimpleModel(
  (fc1): Linear(in_features=10, out_features=50, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=50, out_features=1, bias=True)
)

Predicción de ejemplo: -0.0244


## 2. Guardar y Cargar el Estado del Diccionario (Método 1)

El método más común y flexible para guardar modelos es usando `state_dict`:

In [3]:
# Guardar solo los parámetros del modelo
MODEL_PATH = "model_state_dict.pth"

# Guardar el state_dict del modelo
torch.save(model.state_dict(), MODEL_PATH)
print(f"Modelo guardado en {MODEL_PATH}")

# Cargar el modelo
loaded_model = SimpleModel()
loaded_model.load_state_dict(torch.load(MODEL_PATH))
loaded_model.eval()  # Cambiar a modo evaluación

# Verificar que el modelo cargado produce la misma salida
loaded_output = loaded_model(dummy_input)
print(f"Predicción del modelo original: {dummy_output.item():.4f}")
print(f"Predicción del modelo cargado: {loaded_output.item():.4f}")
print(f"¿Las predicciones coinciden? {torch.allclose(dummy_output, loaded_output)}")

Modelo guardado en model_state_dict.pth
Predicción del modelo original: -0.0244
Predicción del modelo cargado: -0.0244
¿Las predicciones coinciden? True


## 3. Guardar y Cargar el Modelo Completo (Método 2)

También podemos guardar el modelo completo, pero este método es menos flexible:

In [4]:
# Guardar el modelo completo
FULL_MODEL_PATH = "full_model.pth"
torch.save(model, FULL_MODEL_PATH)
print(f"Modelo completo guardado en {FULL_MODEL_PATH}")

# Cargar el modelo completo
loaded_full_model = torch.load(FULL_MODEL_PATH)
loaded_full_model.eval()

# Verificar la salida
loaded_full_output = loaded_full_model(dummy_input)
print(f"Predicción del modelo original: {dummy_output.item():.4f}")
print(f"Predicción del modelo completo cargado: {loaded_full_output.item():.4f}")
print(f"¿Las predicciones coinciden? {torch.allclose(dummy_output, loaded_full_output)}")

Modelo completo guardado en full_model.pth
Predicción del modelo original: -0.0244
Predicción del modelo completo cargado: -0.0244
¿Las predicciones coinciden? True


## 4. Guardar y Cargar Checkpoint (Método 3)

Para entrenamientos, a menudo queremos guardar más que solo el modelo:

In [5]:
# Crear un checkpoint con estado del modelo, optimizador y épocas
CHECKPOINT_PATH = "checkpoint.pth"
epoch = 10
loss = 0.123

# Guardar checkpoint
checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}

torch.save(checkpoint, CHECKPOINT_PATH)
print(f"Checkpoint guardado en {CHECKPOINT_PATH}")

# Cargar checkpoint
checkpoint_loaded = torch.load(CHECKPOINT_PATH)

loaded_checkpoint_model = SimpleModel()
loaded_optimizer = optim.Adam(loaded_checkpoint_model.parameters(), lr=0.001)

loaded_checkpoint_model.load_state_dict(checkpoint_loaded['model_state_dict'])
loaded_optimizer.load_state_dict(checkpoint_loaded['optimizer_state_dict'])
loaded_epoch = checkpoint_loaded['epoch']
loaded_loss = checkpoint_loaded['loss']

loaded_checkpoint_model.eval()

print(f"Época cargada: {loaded_epoch}")
print(f"Pérdida cargada: {loaded_loss:.4f}")

# Verificar predicciones
checkpoint_output = loaded_checkpoint_model(dummy_input)
print(f"Predicción del modelo original: {dummy_output.item():.4f}")
print(f"Predicción del modelo desde checkpoint: {checkpoint_output.item():.4f}")
print(f"¿Las predicciones coinciden? {torch.allclose(dummy_output, checkpoint_output)}")

Checkpoint guardado en checkpoint.pth
Época cargada: 10
Pérdida cargada: 0.1230
Predicción del modelo original: -0.0244
Predicción del modelo desde checkpoint: -0.0244
¿Las predicciones coinciden? True


## 5. Guardar para Inferencia en Producción

Para desplegar modelos en producción, podemos usar TorchScript:

In [6]:
# Guardar modelo para producción usando TorchScript
SCRIPT_MODEL_PATH = "scripted_model.pt"

# Compilar el modelo usando script
scripted_model = torch.jit.script(model)

# Guardar el modelo compilado
scripted_model.save(SCRIPT_MODEL_PATH)
print(f"Modelo compilado guardado en {SCRIPT_MODEL_PATH}")

# Cargar el modelo compilado
loaded_scripted_model = torch.jit.load(SCRIPT_MODEL_PATH)

# Verificar predicción
scripted_output = loaded_scripted_model(dummy_input)
print(f"Predicción del modelo original: {dummy_output.item():.4f}")
print(f"Predicción del modelo compilado: {scripted_output.item():.4f}")
print(f"¿Las predicciones coinciden? {torch.allclose(dummy_output, scripted_output)}")

Modelo compilado guardado en scripted_model.pt
Predicción del modelo original: -0.0244
Predicción del modelo compilado: -0.0244
¿Las predicciones coinciden? True


## 6. Uso Práctico: Guardar el Mejor Modelo Durante el Entrenamiento

Ejemplo de un patrón común durante el entrenamiento:

In [7]:
def train_model(model, epochs=5):
    """Función simplificada para simular entrenamiento"""
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    best_loss = float('inf')
    best_model_path = "best_model.pth"
    
    for epoch in range(epochs):
        # Simular datos de entrenamiento
        inputs = torch.randn(32, 10)
        targets = torch.randn(32, 1)
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass y optimización
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Simular pérdida de validación
        val_loss = loss.item() * 0.9  # Solo para simular
        
        print(f"Época {epoch+1}/{epochs}, Pérdida: {loss.item():.4f}, Pérdida Val: {val_loss:.4f}")
        
        # Guardar el mejor modelo
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            print(f"  Mejor modelo guardado con pérdida: {val_loss:.4f}")
    
    # Cargar el mejor modelo al final del entrenamiento
    model.load_state_dict(torch.load(best_model_path))
    return model

# Crear un nuevo modelo para entrenamiento
training_model = SimpleModel()
trained_model = train_model(training_model, epochs=5)
print("\nEntrenamiento completado y mejor modelo cargado.")

Época 1/5, Pérdida: 0.9819, Pérdida Val: 0.8837
  Mejor modelo guardado con pérdida: 0.8837
Época 2/5, Pérdida: 0.8242, Pérdida Val: 0.7418
  Mejor modelo guardado con pérdida: 0.7418
Época 3/5, Pérdida: 0.9305, Pérdida Val: 0.8375
Época 4/5, Pérdida: 0.9657, Pérdida Val: 0.8691
Época 5/5, Pérdida: 1.2180, Pérdida Val: 1.0962

Entrenamiento completado y mejor modelo cargado.


## 7. Compatibilidad Entre Diferentes Versiones y Dispositivos

Para garantizar la compatibilidad entre diferentes versiones o dispositivos:

In [8]:
# Guardar con compatibilidad entre diferentes dispositivos
COMPATIBLE_PATH = "compatible_model.pth"

# Guardar en CPU incluso si el modelo está en GPU
torch.save(model.state_dict(), COMPATIBLE_PATH)

# Cargar en el dispositivo correcto
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
compatible_model = SimpleModel()
compatible_model.load_state_dict(torch.load(COMPATIBLE_PATH, map_location=device))
compatible_model.to(device)
compatible_model.eval()

print(f"Modelo cargado en dispositivo: {device}")

Modelo cargado en dispositivo: cpu


## 8. Limpieza de los Archivos Creados

Removemos los archivos creados para mantener limpio el directorio:

In [9]:
# Limpiar archivos creados
files_to_remove = [MODEL_PATH, FULL_MODEL_PATH, CHECKPOINT_PATH, 
                  SCRIPT_MODEL_PATH, "best_model.pth", COMPATIBLE_PATH]

for file_path in files_to_remove:
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Archivo eliminado: {file_path}")

Archivo eliminado: model_state_dict.pth
Archivo eliminado: full_model.pth
Archivo eliminado: checkpoint.pth
Archivo eliminado: scripted_model.pt
Archivo eliminado: best_model.pth
Archivo eliminado: compatible_model.pth


## Resumen

En este notebook, hemos visto diferentes métodos para guardar y cargar modelos de PyTorch:

1. **state_dict**: El método más flexible y recomendado para la mayoría de casos.
2. **Modelo completo**: Conveniente pero menos flexible.
3. **Checkpoint**: Ideal para entrenamiento, guarda el estado del modelo y optimizador.
4. **TorchScript**: Para desplegar modelos en producción.

También mostramos ejemplos prácticos de:
- Guardar el mejor modelo durante entrenamiento
- Asegurar compatibilidad entre diferentes dispositivos

La elección del método depende de tu caso de uso específico.